# Preparation

In [ ]:
import os
import numpy as np
import cv2
# import seaborn as sns
import pandas as pd
from astropy.io import fits
from datetime import datetime
import matplotlib.pyplot as plt


from pprint import pprint
from scipy.ndimage import zoom
from skimage.filters import threshold_otsu
from skimage import filters
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from collections import OrderedDict

from captum.attr import LayerGradCam

import sys
current_directory = os.getcwd()# 获取当前脚本所在的目录
sys.path.append(current_directory)# 将该路径添加到 sys.path
from cnn.utils.model import CNN_Model

print(current_directory)

device = torch.device('cuda:0')


In [ ]:

model = CNN_Model(input_channels=4)

model_state = torch.load('./cnn/cnn_model.pth', map_location=device)

# 假设 best_model_state 是 torch.load() 出来的结果
new_state_dict = OrderedDict()
for k, v in model_state.items():
    name = k.replace("module.", "")  # 去掉 `module.` 前缀
    new_state_dict[name] = v

model.load_state_dict(new_state_dict)
model = model.to(device)
model.eval()

# CNN Performance

In [ ]:

metrics_path = os.path.join(current_directory, "cnn", "records", "evaluation_metrics.csv")
metrics_df = pd.read_csv(metrics_path, index_col=0)

# 若存在重复的 dataset 列则去掉
if "dataset" in metrics_df.columns and metrics_df.index.name == "dataset":
    metrics_df = metrics_df.loc[:, metrics_df.columns != "dataset"]

# 数值列保留 4 位小数便于查看
for col in ["recall", "precision", "f1_score", "tss"]:
    if col in metrics_df.columns:
        metrics_df[col] = metrics_df[col].astype(float).round(4)

print("评估指标（来自 evaluation_metrics.csv）:")
display(metrics_df)

# RoIs

In [ ]:
# 读取数据
def read_fits(file_name):
    hdu = fits.open(file_name)
    try:
        data = hdu[1].data
    except:
        data = hdu[0].data
    return data
    
def read_hdu(sample):
    folder_path = "/data_work/data_sharp_cea/"
    try:
        harpnum = sample['HARPNUM'].values[0]
        file_name = sample['file_name'].values[0]
    except:
        harpnum = sample['HARPNUM']
        file_name = sample['file_name']
    file_path = folder_path + f'sharp_{harpnum:05d}/{file_name}'

    bl_data = read_fits(file_path + '.magnetogram.fits')
    br_data = read_fits(file_path + '.Br.fits')
    bp_data = read_fits(file_path + '.Bp.fits')
    bt_data = read_fits(file_path + '.Bt.fits')
    header = pd.read_csv(file_path + '.header.csv')

    data = np.stack([bl_data, br_data, bp_data, bt_data], axis=0)
    return data, header

# ------------------------------ PIL Mask ----------------------------

def get_pil(sample):
    folder_path = "/data_work/mask_data/pil/"
    try:
        harpnum = sample['HARPNUM'].values[0]
        file_name = sample['file_name'].values[0]
    except:
        harpnum = sample['HARPNUM']
        file_name = sample['file_name']
        
    pil_path = folder_path + f'{file_name}.pil.npy'
    pil_data = np.load(pil_path)
    return pil_data

# ------------------------------- SHARP Mask --------------------------

def get_sharpmask(sample):
    folder_path = "/data_work/data_sharp_cea/"
    try:
        harpnum = sample['HARPNUM'].values[0]
        file_name = sample['file_name'].values[0]
    except:
        harpnum = sample['HARPNUM']
        file_name = sample['file_name']
    file_path = folder_path + f'sharp_{harpnum:05d}/{file_name}'
    
    conf_disambig = read_fits(file_path + '.conf_disambig.fits')
    bitmap = read_fits(file_path + '.bitmap.fits')
    sharpmask = (bitmap >= 30) * (conf_disambig >= 70)
    
    return sharpmask, bitmap, conf_disambig

def get_soft_overlay(m_img, b_img, color_code='#87CEEB', alpha=0.9, beta=0.2):
    """ 柔色叠加方案
    Parameters:
        color_code: 十六进制颜色值 (默认淡天蓝色 #87CEEB)
        alpha: 基础图像透明度 (0.0-1.0)
        beta: 叠加层透明度 (0.0-1.0)
    """
    m_img_rgb = plt.cm.gray((m_img >= 70)*255)
    m_img_rgb = (m_img_rgb[:, :, :3] * 255).astype(np.uint8)
    
    # 将HEX转为BGR格式
    rgb = tuple(int(color_code.lstrip('#')[i:i+2], 16) for i in (0, 2, 4))
    bgr_color = (rgb[0], rgb[1], rgb[2])  # 转换RGB到BGR
    
    b_mask = np.zeros_like(m_img_rgb)
    mask_area = (b_img >= 30)
    b_mask[mask_area] = bgr_color

    # 优化gamma值补偿亮度
    overlay = cv2.addWeighted(
        m_img_rgb, alpha, 
        b_mask, beta, 
        gamma=10  # 亮度补偿防止叠加后变暗
    )
    return overlay

# ----------------------- IoU ----------------------------
    
# 数据处理
def pad_matrix_edge3d(A):
    c, h, w = A.shape
    n = max(h, w)  # 新矩阵的大小
    
    # 创建一个大小为 n x n 的矩阵 B，并用0填充
    B = np.zeros((c, n, n), dtype=A.dtype)

    # 将原矩阵 A 的元素填充到 B 的中心
    start_h = (n - h) // 2  # 计算要填充的开始行
    start_w = (n - w) // 2  # 计算要填充的开始列
    
    B[:, start_h:start_h + h, start_w:start_w + w] = A
    
    return B

def ResizeAndNormalize(fits_data):
    c, h, w = fits_data.shape
    max_dim = max(h, w)
    padded_sample = pad_matrix_edge3d(fits_data)

    # 计算变换系数
    resize_factors = 512 / max_dim

    # Resize到512x512
    resized_sample = zoom(padded_sample, (1, resize_factors, resize_factors), order=1)
    
    return resized_sample

def unpad_matrix_edge(B, original_shape):
    # 计算原始矩阵的行列数
    orig_h, orig_w = original_shape
    
    # 计算填充的大小
    padded_h, padded_w = B.shape
    
    # 计算填充的起始位置
    start_x = (padded_h - orig_h) // 2
    start_y = (padded_w - orig_w) // 2
    
    # 剪切出原始矩阵
    unpadded_matrix = B[start_x:start_x + orig_h, start_y:start_y + orig_w]
    
    return unpadded_matrix

def data_restore(resized_sample, original_shape):
    # 计算变换系数（反向）
    max_dim = max(original_shape)
    resize_factors = resized_sample.shape[0] / max_dim

    # 反向缩放
    unresized_sample = zoom(resized_sample, (1 / resize_factors, 1 / resize_factors), order=1)

    # 去除填充
    unpadded_sample = unpad_matrix_edge(unresized_sample, original_shape)

    return unpadded_sample

def get_gradcam_heatmap(fits_data, model, device):

    # input_data preprocessing
    processed_data = ResizeAndNormalize(fits_data)
    input_data = torch.tensor(processed_data).unsqueeze(0).float().to(device)

    # Grad-CAM
    # res_blocks[6].conv2; third_conv; conv3
    layer_gc = LayerGradCam(model, model.res_blocks[-2].conv2)
    attr_gc = layer_gc.attribute(input_data, target=0)
    attr_gc = attr_gc.cpu().detach().numpy().squeeze().squeeze()

    # output hitmap processing
    resized_attr = data_restore(attr_gc, (fits_data.shape[1], fits_data.shape[2]))
    
    return resized_attr, nn.Sigmoid()(model(input_data)).cpu().detach().numpy().squeeze().squeeze()

# 通过 OTSU 对 attr_data 进行阈值分割得到 IoU
def get_iou(heatmap):
    # 计算阈值
    threshold = filters.threshold_otsu(np.abs(heatmap))
    
    # 创建掩膜
    mask = np.zeros_like(heatmap, dtype=int)
    mask[heatmap > threshold] = 1
    mask[heatmap < -threshold] = -1
    
    return mask


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_results(
    base_img,
    sample_attr,
    conf_bitmap,
    sample_pil,
    sample_iou,
    bitmap,
    conf_disambig,
    dpi=100,
    save_path=None
):
    """
    绘制 2x3 子图布局的可视化结果
    
    参数
    ----------
    base_img : np.ndarray
        基础灰度图像
    sample_attr : np.ndarray
        归一化特征属性图
    conf_bitmap : np.ndarray
        置信度位图
    sample_pil : np.ndarray
        PIL mask
    sample_iou : np.ndarray
        IoU mask
    bitmap : np.ndarray
        原始位图
    conf_disambig : np.ndarray
        去歧义置信度图
    dpi : int, 可选
        绘图分辨率，默认 100
    save_path : str, 可选
        如果指定，则保存为 svg 文件
    """
    
    # 创建画布
    fig_width = 900 * 3 / dpi + 2
    fig_height = 400 * 2 / dpi + 0.5
    fig = plt.figure(figsize=(fig_width, fig_height))
    
    # 创建2x3网格
    gs = fig.add_gridspec(2, 3, wspace=0.05, hspace=0.02)

    # ax1
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(base_img, vmin=-1500, vmax=1500, cmap='gray', origin="lower")

    # ax2
    ax2 = fig.add_subplot(gs[0, 1])
    im = ax2.imshow(
        sample_attr / np.abs(sample_attr).max(),
        vmin=-1,
        vmax=1,
        cmap='jet',
        origin='lower'
    )
    divider = make_axes_locatable(ax2)
    cax = divider.append_axes("right", size=0.15, pad=0.1)
    cbar = fig.colorbar(im, cax=cax)
    cbar.set_ticks([-1, 0, 1])
    cbar.set_ticklabels(['-1', '0', '1'])
    cbar.ax.tick_params(labelsize=24)

    # ax3
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(conf_bitmap, origin="lower")

    # ax4
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.imshow(base_img, vmin=-1500, vmax=1500, cmap='gray', origin="lower")
    ax4.contour(sample_pil, colors='#0dff76', linewidths=1)

    # ax5
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.imshow(base_img, vmin=-1500, vmax=1500, cmap='gray', origin="lower")
    ax5.contour(sample_iou, colors='#0d80ff', linewidths=1.5)

    # ax6
    ax6 = fig.add_subplot(gs[1, 2])
    ax6.imshow(base_img, vmin=-1500, vmax=1500, cmap='gray', origin="lower")
    ax6.contour((bitmap >= 30) * (conf_disambig >= 70), colors='#ff2b19', linewidths=1)

    # 设置子图标签
    labels = ['a', 'b', 'c', 'd', 'e', 'f']
    for idx, ax in enumerate(fig.axes):
        ax.tick_params(axis='both', which='major', labelsize=10)
        ax.grid(True, alpha=0.3)
        if idx == 2:  # 跳过 ax3 的坐标轴
            continue
        elif idx > 2:
            idx -= 1
            
        ax.set_xticks([])
        ax.set_yticks([])
        text_color = 'white' if idx == 2 else 'black'
        ax.text(
            0.02, 0.85, labels[idx], 
            transform=ax.transAxes,
            fontsize=32,
            fontweight='bold',
            color=text_color
        )

    # 布局调整
    plt.tight_layout()
    
    # 保存或显示
    if save_path:
        plt.savefig(save_path, format='svg', bbox_inches='tight', pad_inches=0.1)
    else:
        plt.show()


In [ ]:

test_sample = {"HARPNUM": 5983, "file_name": 'hmi.sharp_cea_720s.5983.20150930_030000_TAI'}
sample_data, sample_header = read_hdu(test_sample)

sample_blos = sample_data[0]

# get sharp_mask
sample_sharp, bitmap, conf_disambig = get_sharpmask(test_sample)
conf_bitmap = get_soft_overlay(conf_disambig, bitmap, color_code='#FF2B19', alpha=1.0, beta=0.8)

# get_pil
sample_pil = get_pil(test_sample)

plt.figure()
plt.imshow(sample_blos, vmin=-1500, vmax=1500, cmap='gray', origin='lower')
plt.axis('off')
plt.show()

In [ ]:
# compute iou
sample_attr,pred = get_gradcam_heatmap(sample_data, model, device)
print(pred)

sample_iou = get_iou(sample_attr)

plot_results(
    base_img=sample_data[0],
    sample_attr=sample_attr,
    conf_bitmap=conf_bitmap,
    sample_pil=sample_pil,
    sample_iou=sample_iou,
    bitmap=bitmap,
    conf_disambig=conf_disambig,
    save_path=None
)


# figure samples

In [ ]:

def plot_gradcam_examples(select_df, model, device, read_hdu, get_gradcam_heatmap, get_iou, labels=None):
    """
    绘制 6 行 3 列的 Grad-CAM 示例图，每行对应一个样本，每列显示不同内容：
    1. 原始 Blos 图
    2. Grad-CAM 热图
    3. Grad-CAM 掩码叠加图

    每行高度自适应，每列宽度固定，行列间距可调。
    保留边框，不显示刻度和标签，并在每行第一个子图左上角标注 a~f。
    """
    n_rows = len(select_df)
    n_cols = 3
    
    # 默认 labels 为 a~f
    if labels is None:
        labels = [chr(ord('a') + i) for i in range(n_rows)]
    
    # 计算每行高度比例（保持列宽固定，高度自适应）
    height_ratios = []
    for i in range(n_rows):
        sample = select_df.iloc[i]
        sample_data, _ = read_hdu(sample)
        h, w = sample_data[0].shape
        height_ratios.append(h / w)
    
    # 创建 figure 和 GridSpec
    fig = plt.figure(figsize=(15, sum(height_ratios) * 5.5))
    gs = GridSpec(n_rows, n_cols, height_ratios=height_ratios, hspace=0.1, wspace=0.05)
    
    for i, sample in select_df.iterrows():
        sample_data, sample_header = read_hdu(sample)
        sample_heatmap, sample_prediction = get_gradcam_heatmap(sample_data, model, device)
        print(f"{sample['file_name']}, {sample['label']}, {sample_prediction.item():.3f}, {sample['entire pii_pos']:.3f}, {sample['entire pii_neg']:.3f}")
        sample_heatmap = sample_heatmap / np.max(np.abs(sample_heatmap))
        sample_mask = get_iou(sample_heatmap)
        
        # 第 1 列：Blos 图
        ax0 = fig.add_subplot(gs[i, 0])
        ax0.imshow(sample_data[0], vmin=-1500, vmax=1500, cmap='gray', origin='lower')
        # 隐藏刻度和标签，但保留边框
        ax0.set_xticks([])
        ax0.set_yticks([])
        # 左上角标注
        ax0.text(0.02, 0.95, labels[i], transform=ax0.transAxes,
                 fontsize=24, fontweight='bold', va='top', ha='left')
        
        # 第 2 列：Grad-CAM 热图
        ax1 = fig.add_subplot(gs[i, 1])
        ax1.imshow(sample_heatmap, vmin=-1, vmax=1, cmap='bwr', origin='lower')
        ax1.set_xticks([])
        ax1.set_yticks([])
        
        # 第 3 列：Grad-CAM 掩码叠加图
        ax2 = fig.add_subplot(gs[i, 2])
        ax2.imshow(sample_data[0], vmin=-1500, vmax=1500, cmap='gray', origin='lower')
        ax2.contour(sample_mask == 1, colors='r', linewidths=1)
        ax2.contour(sample_mask == -1, colors='b', linewidths=1)
        ax2.set_xticks([])
        ax2.set_yticks([])
    
    plt.show()


In [ ]:
select_file = [
'hmi.sharp_cea_720s.5983.20150930_030000_TAI',
'hmi.sharp_cea_720s.2693.20130502_080000_TAI',
'hmi.sharp_cea_720s.1119.20111201_150000_TAI',
'hmi.sharp_cea_720s.2748.20130522_080000_TAI',
'hmi.sharp_cea_720s.504.20110416_080000_TAI',
'hmi.sharp_cea_720s.6020.20151014_020000_TAI']

label_df = pd.read_csv("./data/label.csv")

select_df = label_df[label_df['file_name'].isin(select_file)]

# 将 'file_name' 列转换为分类类型，并指定顺序
select_df['file_name'] = pd.Categorical(
    select_df['file_name'], 
    categories=select_file, 
    ordered=True
)

# 按 'file_name' 排序
select_df = select_df.sort_values('file_name')

select_df = select_df.reset_index(drop=True)

pii_df = pd.read_csv("./data/PII_records.csv")
select_df = select_df.merge(pii_df, on=['file_name', 'label'], how='left')

# 调用封装函数绘图
plot_gradcam_examples(select_df, model, device, read_hdu, get_gradcam_heatmap, get_iou)

# Figure PII Distribution

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd

# ========== 数据读取 ==========
csv_file = './data/PII_records.csv'
df = pd.read_csv(csv_file)
df = df.rename(columns={
    'weighted pii_pos': 'PII_p',
    'weighted pii_neg': 'PII_n'
})
df.dropna(subset=["PII_p", "PII_n"], inplace=True)

# 自定义颜色（KDE线用深色，散点用浅色）
label_colors = {
    0: {"kde": "#0d80ff", "scatter": "#a6c8ff"},  # 蓝 + 浅蓝
    1: {"kde": "#ff2b19", "scatter": "#ffb3a6"}   # 红 + 浅红
}

# ========== 主图 ==========
fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(111)

for label, colors in label_colors.items():
    mask = (df["label"] == label)
    if sum(mask) < 2:  # KDE 需要至少 2 个点
        continue

    x = df.loc[mask, "PII_p"].values
    y = df.loc[mask, "PII_n"].values

    # 计算 KDE（限定范围 -0.2 到 1.2）
    kde = gaussian_kde(np.vstack([x, y]))
    xx, yy = np.mgrid[-0.2:1.2:100j, -0.2:1.2:100j]
    zz = kde(np.vstack([xx.ravel(), yy.ravel()]))

    # 设置等高线层级和透明度
    levels = np.linspace(zz.min(), zz.max(), 6)
    alphas = np.linspace(0., 1.0, len(levels))

    # 绘制等高线 + 标注
    for level, alpha in zip(levels, alphas):
        if alpha == 0.2:
            continue
        contour = ax.contour(xx, yy, zz.reshape(xx.shape),
                            levels=[level],
                            colors=[colors["kde"]],
                            alpha=1,
                            linewidths=2)
        
        # 输出每个 contour 的 x, y 范围
        for i, seg in enumerate(contour.allsegs[0]):  # contour.allsegs[0] 对应当前 level
            if seg.size == 0:
                continue  # 跳过空 contour
            x_coords = seg[:, 0]
            y_coords = seg[:, 1]
            x_min, x_max = x_coords.min(), x_coords.max()
            y_min, y_max = y_coords.min(), y_coords.max()
            print(f"Label {label}, level {alpha:.4f}, contour {i}: "
                f"x=[{x_min:.3f}, {x_max:.3f}], y=[{y_min:.3f}, {y_max:.3f}]")

        fmt_dict = {level: f"{alpha:.2f}"}
        ax.clabel(contour, inline=True, fontsize=24, fmt=fmt_dict)

    # 绘制散点（浅色，不透明）
    ax.scatter(x, y, color=colors["scatter"], s=36, alpha=1, label=f"Samples {label}")

    # 峰值点
    peak_idx = np.argmax(zz)
    peak_x, peak_y = xx.ravel()[peak_idx], yy.ravel()[peak_idx]
    print(f"KDE图： Label {label}: PII 峰值位置 -> x={peak_x:.3f}, y={peak_y:.3f}")

    # x, y 已经是当前 label 的数据
    H, xedges, yedges = np.histogram2d(x, y, bins=50, range=[[0, 1], [0, 1]])

    # 找到峰值 bin 的索引
    peak_idx = np.unravel_index(np.argmax(H), H.shape)
    peak_x = (xedges[peak_idx[0]] + xedges[peak_idx[0]+1]) / 2
    peak_y = (yedges[peak_idx[1]] + yedges[peak_idx[1]+1]) / 2

    print(f"直方图： Label {label}: Histogram 峰值位置 -> x={peak_x:.3f}, y={peak_y:.3f}")

# 设置主图坐标轴范围和标签
ax.set_xlim(-0.2, 1.2)
ax.set_ylim(-0.2, 1.2)
ax.set_xlabel("PII in positive activation area", fontsize=48, labelpad=24)
ax.set_ylabel("PII in negative activation area", fontsize=48, labelpad=16)
ax.tick_params(axis='both', which='major', labelsize=32)
ax.tick_params(axis='both', which='minor', labelsize=32)

# ========== 边际图 ==========
divider = make_axes_locatable(ax)

# 顶部边际图
ax_marg_x = divider.append_axes("top", size="20%", pad=0.1)
for label, colors in label_colors.items():
    mask = (df["label"] == label)
    if sum(mask) < 2:
        continue
        
    x_data = df.loc[mask, "PII_p"].values
    
    # 使用 gaussian_kde 计算边际分布
    if len(x_data) > 1:
        kde_x = gaussian_kde(x_data)
        x_range = np.linspace(-0.2, 1.2, 200)
        density_x = kde_x(x_range)
        
        # 绘制填充的密度图
        ax_marg_x.fill_between(x_range, density_x, alpha=0.5, color=colors["kde"])
        ax_marg_x.plot(x_range, density_x, color=colors["kde"], alpha=0.8)

ax_marg_x.set_xlim(-0.2, 1.2)
ax_marg_x.set_ylim(0, ax_marg_x.get_ylim()[1])
ax_marg_x.set_xticks([])
ax_marg_x.set_yticks([])
ax_marg_x.set_ylabel("Density", fontsize=32)
ax_marg_x.set_xlabel("")
ax_marg_x.tick_params(axis='both', labelsize=16)

# 右侧边际图
ax_marg_y = divider.append_axes("right", size="20%", pad=0.1)
for label, colors in label_colors.items():
    mask = (df["label"] == label)
    if sum(mask) < 2:
        continue
        
    y_data = df.loc[mask, "PII_n"].values
    
    # 使用 gaussian_kde 计算边际分布
    if len(y_data) > 1:
        kde_y = gaussian_kde(y_data)
        y_range = np.linspace(-0.2, 1.2, 200)
        density_y = kde_y(y_range)
        
        # 绘制填充的密度图
        ax_marg_y.fill_betweenx(y_range, density_y, alpha=0.5, color=colors["kde"])
        ax_marg_y.plot(density_y, y_range, color=colors["kde"], alpha=0.8)

ax_marg_y.set_ylim(-0.2, 1.2)
ax_marg_y.set_xlim(0, ax_marg_y.get_xlim()[1])
ax_marg_y.set_xticks([])
ax_marg_y.set_yticks([])
ax_marg_y.set_xlabel("Density", fontsize=32)
ax_marg_y.set_ylabel("")
ax_marg_y.tick_params(axis='both', labelsize=16)

# 添加图例
label_list = ["negative events", "positive events"]
handles = [plt.Line2D([], [], color=colors["kde"], label=label_list[label])
           for label, colors in label_colors.items()]
ax.legend(handles=handles, fontsize=36, loc="lower right")

plt.tight_layout()
plt.show()


# Parameters Evaluation

In [ ]:
import glob
from operator import index
# 读取并合并所有数据
file_paths = glob.glob("./param/a2/perf/*.csv")
all_data = []

for fp in file_paths:
    dataset = fp.split("/")[-1].replace(".csv", "")
    print(dataset)
    df = pd.read_csv(fp)
    df['dataset'] = dataset
    all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)
combined_df['specificity'] = combined_df['tn'] / (combined_df['tn'] + combined_df['fp'])

# 定义需要统计的指标列
metric_cols = ['tp', 'fp', 'tn', 'fn', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'tss']

# 分组计算统计量
grouped = combined_df.groupby(['model', 'dataset'])[metric_cols]
stats_df = grouped.agg(['mean', 'std'])

# 处理统计结果（合并为mean ± std格式）
formatted_data = []
for (model, dataset), group_data in grouped:
    formatted_row = {'Model': model, 'Dataset': dataset}
    
    for metric in metric_cols:
        mean_val = group_data[metric].mean()
        std_val = group_data[metric].std()
        
        # 处理可能的NaN（当只有1个样本时）
        if pd.isna(std_val):
            std_val = 0.0
        
        # 特殊处理训练时间格式
        if metric == 'training_time':
            formatted = f"{mean_val:.1f} ± {std_val:.1f} s"
        else:
            formatted = f"{mean_val:.5f} "# ± {std_val:.3f}
        
        formatted_row[metric.capitalize()] = formatted
    
    formatted_data.append(formatted_row)

# 生成最终表格
final_df = pd.DataFrame(formatted_data).set_index(['Model', 'Dataset'])

# 打印精美表格
styled = final_df.style.set_properties(**{
    'text-align': 'center',
    'white-space': 'pre'
}).set_table_styles([{
    'selector': 'th',
    'props': [('background-color', '#f7f7f7'), ('font-weight', 'bold')]
}])
print("模型性能汇总表（Mean）：")
display(styled)

out_csv = "./data//performance_summary_a2.csv"
final_df.to_csv(out_csv, encoding="utf-8-sig")

In [ ]:
import glob
# 读取并合并所有数据
file_paths = glob.glob("./param/a3/perf/*.csv")
all_data = []

for fp in file_paths:
    print(fp)
    dataset = fp.split("/")[-1].replace(".csv", "")
    print(dataset)
    df = pd.read_csv(fp)
    df['dataset'] = dataset
    all_data.append(df)

combined_df = pd.concat(all_data, ignore_index=True)
combined_df['specificity'] = combined_df['tn'] / (combined_df['tn'] + combined_df['fp'])

# 定义需要统计的指标列
metric_cols = ['tp', 'fp', 'tn', 'fn', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'tss']

# 分组计算统计量
grouped = combined_df.groupby(['model', 'dataset'])[metric_cols]
stats_df = grouped.agg(['mean', 'std'])

# 处理统计结果（合并为mean ± std格式）
formatted_data = []
for (model, dataset), group_data in grouped:
    formatted_row = {'Model': model, 'Dataset': dataset}
    
    for metric in metric_cols:
        mean_val = group_data[metric].mean()
        std_val = group_data[metric].std()
        
        # 处理可能的NaN（当只有1个样本时）
        if pd.isna(std_val):
            std_val = 0.0
        
        # 特殊处理训练时间格式
        if metric == 'training_time':
            formatted = f"{mean_val:.1f} ± {std_val:.1f} s"
        else:
            formatted = f"{mean_val:.5f}" #  ± {std_val:.3f}
        
        formatted_row[metric.capitalize()] = formatted
    
    formatted_data.append(formatted_row)

# 生成最终表格
final_df = pd.DataFrame(formatted_data).set_index(['Model', 'Dataset'])

# 打印精美表格
styled = final_df.style.set_properties(**{
    'text-align': 'center',
    'white-space': 'pre'
}).set_table_styles([{
    'selector': 'th',
    'props': [('background-color', '#f7f7f7'), ('font-weight', 'bold')]
}])
print("模型性能汇总表（Mean）：")
display(styled)

out_csv = "./data//performance_summary_a3.csv"
final_df.to_csv(out_csv, encoding="utf-8-sig")